Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC -- NC_Dump: paste into sheet "NC_Dump" (row 1 = headers, row 2+ = data).
# MAGIC -- Column order A–L must match exactly (Apps Script NC_COL indices).
# MAGIC
# MAGIC --  A  customer_id
# MAGIC --  B  canonical_ticket_id
# MAGIC --  C  ticket_dt
# MAGIC --  D  first_call_date_timestamp
# MAGIC --  E  total_calls_made_on_ticket
# MAGIC --  F  calls_day_0
# MAGIC --  G  calls_day_1
# MAGIC --  H  calls_day_2
# MAGIC --  I  agent_name
# MAGIC --  J  bsd_tier
# MAGIC --  K  offer_max_loan_amount
# MAGIC --  L  call_creation_timestamps   (JSON array — used for unique call-days)
# MAGIC
# MAGIC WITH nc_tickets AS (
# MAGIC     SELECT
# MAGIC         canonical_ticket_id,
# MAGIC         customer_id,
# MAGIC         CAST(first_ticket_created_date AS DATE) AS ticket_dt,
# MAGIC         CAST(first_call_timestamp AS TIMESTAMP) AS first_call_ts,
# MAGIC         total_calls,
# MAGIC         agents_involved,
# MAGIC         call_creation_timestamps
# MAGIC     FROM corp_analytics.pl_apex_canonical_tickets_tbl
# MAGIC     WHERE disposition_reason = 'Not Connected'
# MAGIC       AND DATE(first_ticket_created_date)
# MAGIC             BETWEEN CURRENT_DATE - INTERVAL '15' DAY AND CURRENT_DATE
# MAGIC       AND total_calls <= 6
# MAGIC       AND chat_ticket_state NOT IN ('FORCE_CLOSED', 'RESOLVED')
# MAGIC       AND disbursal_flag = 0
# MAGIC ),
# MAGIC
# MAGIC cust_bsd AS (
# MAGIC     SELECT customer_id, bsd_tier
# MAGIC     FROM (
# MAGIC         SELECT
# MAGIC             o.customer_reference_id AS customer_id,
# MAGIC             TRY_CAST(s.bsdv1_retrain_tier AS INTEGER) AS bsd_tier,
# MAGIC             ROW_NUMBER() OVER (
# MAGIC                 PARTITION BY o.customer_reference_id
# MAGIC                 ORDER BY o.loan_offer_reference_id DESC
# MAGIC             ) AS rn
# MAGIC         FROM corp_analytics.customer_last_approved_offer_tbl o
# MAGIC         INNER JOIN corp_analytics.pl_uw_scores_tiers_tbl s
# MAGIC             ON s.loan_offer_reference_id = o.loan_offer_reference_id
# MAGIC         WHERE TRY_CAST(s.bsdv1_retrain_tier AS INTEGER) BETWEEN 1 AND 10
# MAGIC     ) ranked
# MAGIC     WHERE rn = 1
# MAGIC ),
# MAGIC
# MAGIC latest_active_stock AS (
# MAGIC     SELECT
# MAGIC         t.canonical_ticket_id,
# MAGIC         s.customer_reference_id,
# MAGIC         s.offer_max_loan_amount,
# MAGIC         s.expiry_date,
# MAGIC         DATE(SUBSTRING(s.stock_date, 1, 10)) AS stock_dt,
# MAGIC         ROW_NUMBER() OVER (
# MAGIC             PARTITION BY t.canonical_ticket_id
# MAGIC             ORDER BY DATE(SUBSTRING(s.stock_date, 1, 10)) DESC
# MAGIC         ) AS rn
# MAGIC     FROM nc_tickets t
# MAGIC     INNER JOIN corp_analytics.pl_stock_tbl s
# MAGIC         ON t.customer_id = s.customer_reference_id
# MAGIC        AND DATE(SUBSTRING(s.stock_date, 1, 10)) <= t.ticket_dt
# MAGIC        AND DATE(s.expiry_date) >= t.ticket_dt
# MAGIC        AND DATE(SUBSTRING(s.stock_date, 1, 10)) >= CURRENT_DATE - INTERVAL '6' MONTH
# MAGIC ),
# MAGIC
# MAGIC eligible_tickets AS (
# MAGIC     SELECT
# MAGIC         t.*,
# MAGIC         st.offer_max_loan_amount,
# MAGIC         COALESCE(cb.bsd_tier, -1) AS bsd_tier
# MAGIC     FROM nc_tickets t
# MAGIC     INNER JOIN latest_active_stock st
# MAGIC         ON t.canonical_ticket_id = st.canonical_ticket_id
# MAGIC        AND st.rn = 1
# MAGIC     LEFT JOIN cust_bsd cb
# MAGIC         ON t.customer_id = cb.customer_id
# MAGIC     WHERE st.offer_max_loan_amount >= 400000
# MAGIC ),
# MAGIC
# MAGIC call_level AS (
# MAGIC     SELECT
# MAGIC         t.canonical_ticket_id,
# MAGIC         t.customer_id,
# MAGIC         t.ticket_dt,
# MAGIC         t.first_call_ts,
# MAGIC         t.total_calls,
# MAGIC         t.agents_involved,
# MAGIC         t.offer_max_loan_amount,
# MAGIC         t.bsd_tier,
# MAGIC         t.call_creation_timestamps,
# MAGIC         CAST(call_ts AS TIMESTAMP) AS call_ts
# MAGIC     FROM eligible_tickets t
# MAGIC     LEFT JOIN UNNEST(
# MAGIC         CAST(json_parse(t.call_creation_timestamps) AS ARRAY(VARCHAR))
# MAGIC     ) AS u(call_ts)
# MAGIC     ON TRUE
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     customer_id,
# MAGIC     canonical_ticket_id,
# MAGIC     ticket_dt,
# MAGIC     first_call_ts                AS first_call_date_timestamp,
# MAGIC     total_calls                  AS total_calls_made_on_ticket,
# MAGIC
# MAGIC     COUNT(CASE WHEN DATE(call_ts) = ticket_dt                     THEN 1 END) AS calls_day_0,
# MAGIC     COUNT(CASE WHEN DATE(call_ts) = ticket_dt + INTERVAL '1' DAY THEN 1 END) AS calls_day_1,
# MAGIC     COUNT(CASE WHEN DATE(call_ts) = ticket_dt + INTERVAL '2' DAY THEN 1 END) AS calls_day_2,
# MAGIC
# MAGIC     agents_involved              AS agent_name,
# MAGIC     bsd_tier,
# MAGIC     offer_max_loan_amount,
# MAGIC     MAX(call_creation_timestamps) AS call_creation_timestamps
# MAGIC
# MAGIC FROM call_level
# MAGIC GROUP BY
# MAGIC     customer_id,
# MAGIC     canonical_ticket_id,
# MAGIC     ticket_dt,
# MAGIC     first_call_ts,
# MAGIC     total_calls,
# MAGIC     agents_involved,
# MAGIC     bsd_tier,
# MAGIC     offer_max_loan_amount
# MAGIC ORDER BY first_call_date_timestamp
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/17-FP7_n32hhr88Nb38WzqGYuxazUulzdQfmW2D41o4E/edit?gid=1325658883#gid=1325658883"
worksheet_name = "Not Connected Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)